In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[2]
sys.path.append(str(PROJECT_ROOT / "03_codigo" / "utils"))

from utils import *

In [ ]:
# ===== CONFIG GENERAL =====
DATASET = "BCCC17"

ESTADO_ENTRADA = "split"
VERSION_ENTRADA = "v1"

ESTADO_SALIDA = "NearMiss_SMOTE_ENN"
VERSION_SALIDA = "v1"

LABEL_COL = "LABEL"

TARGET_N = 10000
RANDOM_STATE = 42

# ===== NOMBRES DE SPLITS =====
SPLIT_TRAIN = "train"
SPLIT_TEST = "test"

In [ ]:
def construir_nombre_dataset(dataset, estado, version):
    return f"{dataset}__{estado}__{version}"

def construir_nombre_csv(nombre_dataset, split):
    return f"{nombre_dataset}__{split}.csv"

def construir_ruta_base_processed(project_root, nombre_dataset):
    return project_root / "02_datasets" / "processed" / nombre_dataset

def construir_ruta_base_modelos_simples(project_root, nombre_dataset):
    return project_root / "02_datasets" / "processed" / nombre_dataset

In [ ]:
nombre_dataset_entrada = construir_nombre_dataset(DATASET, ESTADO_ENTRADA, VERSION_ENTRADA)
nombre_dataset_salida = construir_nombre_dataset(DATASET, ESTADO_SALIDA, VERSION_SALIDA)

nombre_train = construir_nombre_csv(nombre_dataset_entrada, SPLIT_TRAIN)
nombre_test = construir_nombre_csv(nombre_dataset_entrada, SPLIT_TEST)

nombre_train_generado = construir_nombre_csv(nombre_dataset_salida, SPLIT_TRAIN)
nombre_test_generado = construir_nombre_csv(nombre_dataset_salida, SPLIT_TEST)
nombre_test_generado_extra = nombre_test_generado.replace(".csv", "__nuevo.csv")

ruta_base_dataset = construir_ruta_base_processed(PROJECT_ROOT, nombre_dataset_entrada)
ruta_base_dataset_generado = construir_ruta_base_modelos_simples(PROJECT_ROOT, nombre_dataset_salida)

print("=== ENTRADA ===")
print("Dataset:", nombre_dataset_entrada)
print("Train  :", nombre_train)
print("Test   :", nombre_test)
print("Ruta   :", ruta_base_dataset)
print()

print("=== SALIDA ===")
print("Dataset:", nombre_dataset_salida)
print("Train  :", nombre_train_generado)
print("Test   :", nombre_test_generado)
print("Test extra:", nombre_test_generado_extra)
print("Ruta   :", ruta_base_dataset_generado)

In [ ]:
df_train = cargar_dataset(nombre_train, ruta_base_dataset)
df_test = cargar_dataset(nombre_test, ruta_base_dataset)

In [ ]:
print("Distribución inicial TRAIN:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

print("Distribución inicial TEST:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

In [ ]:
df_train_balanceado, df_test_nuevo = balancear_nearmiss_y_mover_a_test(
    df_train, 
    df_test, 
    LABEL_COL, 
    TARGET_N,
    random_state=RANDOM_STATE,
)

df_train_balanceado = aplicar_enn(df_train_balanceado)

In [ ]:
print("Distribución final TRAIN balanceado:")
display(df_train_balanceado[LABEL_COL].value_counts(dropna=False).to_frame("count"))

print("Distribución final TEST original:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

print("Distribución final TEST nuevo:")
display(df_test_nuevo[LABEL_COL].value_counts(dropna=False).to_frame("count"))

In [ ]:
Path(ruta_base_dataset_generado).mkdir(parents=True, exist_ok=True)

print("Ruta de salida creada/verificada:")
print(Path(ruta_base_dataset_generado).resolve())

In [ ]:
guardar_dataset_csv(df_train_balanceado, nombre_train_generado, ruta_base_dataset_generado)
guardar_dataset_csv(df_test, nombre_test_generado, ruta_base_dataset_generado)
guardar_dataset_csv(df_test_nuevo, nombre_test_generado_extra, ruta_base_dataset_generado)

print("Datasets guardados correctamente.")
print()
print("Train balanceado :", Path(ruta_base_dataset_generado) / nombre_train_generado)
print("Test original    :", Path(ruta_base_dataset_generado) / nombre_test_generado)
print("Test nuevo       :", Path(ruta_base_dataset_generado) / nombre_test_generado_extra)

In [ ]:
print("========== RESUMEN ==========")
print("Dataset entrada :", nombre_dataset_entrada)
print("Dataset salida  :", nombre_dataset_salida)
print()
print("Train original  :", df_train.shape)
print("Test original   :", df_test.shape)
print("Train balanceado:", df_train_balanceado.shape)
print("Test nuevo      :", df_test_nuevo.shape)
print()
print("Parámetros usados:")
print("LABEL_COL       =", LABEL_COL)
print("TARGET_N        =", TARGET_N)
print("RANDOM_STATE    =", RANDOM_STATE)